<a href="https://colab.research.google.com/github/bhamees79-alt/SIH_PROJECT/blob/main/BAS_YOLOv8_Clean_Split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_dataset.zip to clean_dataset.zip


In [ ]:
!unzip -q clean_dataset.zip -d /content/

In [ ]:
!ls -la /content/clean_dataset


total 24
drwxr-xr-x 5 root root 4096 Sep  6 10:11 .
drwxr-xr-x 1 root root 4096 Sep  6 10:11 ..
-rw-r--r-- 1 root root  378 Sep  6  2026 data.yaml
drw-r--r-- 4 root root 4096 Sep  6  2026 test
drw-r--r-- 4 root root 4096 Sep  6  2026 train
drw-r--r-- 4 root root 4096 Sep  6  2026 valid


In [ ]:
import os
import yaml

# 1. Grant full read & enter permissions to all extracted folders and files
!chmod -R 755 /content/clean_dataset

# 2. Point data.yaml explicitly to the Colab paths
yaml_path = '/content/clean_dataset/data.yaml'

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['path'] = '/content/clean_dataset'
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.safe_dump(config, f)

print("✅ Permissions fixed and data.yaml configured:")
!cat /content/clean_dataset/data.yaml

✅ Permissions fixed and data.yaml configured:
names:
- cotton
- dropper
- seed_jar
- target_plate
- water_dish
nc: 5
path: /content/clean_dataset
roboflow:
  license: CC BY 4.0
  project: my-first-projectbas-methi-experi
  url: https://universe.roboflow.com/samriddhi-srivastava-secbi/my-first-projectbas-methi-experi/dataset/1
  version: 1
  workspace: samriddhi-srivastava-secbi
test: test/images
train: train/images
val: valid/images


In [ ]:
from ultralytics import YOLO

# Load base model
model = YOLO('yolov8n.pt')

# Train for 50 epochs on the clean dataset
results = model.train(
    data='/content/clean_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    fliplr=0.5,      # Horizontal flip left-to-right
    mosaic=1.0,      # Dynamic 4-image mosaic
    name='methi_leakfree_run'
)

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/clean_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=methi_leakfree_run, nbs=64, nms=None, opset=

In [ ]:
from google.colab import files
files.download('/content/runs/detect/methi_leakfree_run/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO

# Load the newly trained clean best weights
model = YOLO('/content/runs/detect/methi_leakfree_run/weights/best.pt')

# Run validation against the untouched test split
test_results = model.val(
    data='/content/clean_dataset/data.yaml',
    split='test'
)


Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1089.0±410.8 MB/s, size: 36.2 KB)
val: Scanning /content/clean_dataset/test/labels... 78 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 78/78 1.4Kit/s 0.1s
val: New cache created: /content/clean_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
                   all         78        421      0.931      0.843      0.876      0.752
                cotton         76         78      0.928      0.936      0.948      0.785
               dropper         75         90      0.882       0.75      0.781      0.494
              seed_jar         78         79       0.98      0.962       0.96      0.942
          target_plate         74         95      0.868      0.579      0.699  